In [1]:
import os
import pandas as pd
import re
from datetime import datetime, timedelta
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup

In [3]:
# ==========================================
# 1. 크롤링 및 CSV 저장 함수
# ==========================================
def crawl_and_save_csv(product_id):
    print(f"🚀 [{product_id}] 크롬 브라우저를 준비합니다...")
    
    # 🌟 [핵심 패치 1] 드라이버를 설치하고 그 '정확한 위치(절대 경로)'를 알아냅니다.
    driver_path = ChromeDriverManager().install()
    
    # 🌟 [핵심 패치 2] 알아낸 위치의 파일에 직접 '보안 격리 해제' 명령을 때립니다!
    os.system(f"xattr -cr '{driver_path}'")
    
    print(f"🔓 맥 OS 보안 차단 완벽 해제! (경로: {driver_path})")
    
    options = webdriver.ChromeOptions()
    # 크림 접속을 위한 완벽한 신분증(User-Agent)
    options.add_argument("user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")
    options.add_experimental_option("detach", True)
    
    # 차단이 풀린 드라이버를 안심하고 실행합니다.
    driver = webdriver.Chrome(service=Service(driver_path), options=options)
    
    url = f"https://kream.co.kr/products/{product_id}"
    driver.get(url)
    
    print("\n" + "🔥"*20)
    print("🚨 [수동 작업 안내]")
    print("1. 팝업창을 닫아주세요.")
    print("2. '체결 내역' 창을 클릭해주세요.")
    print("3. 마우스 휠을 밑으로 쫙쫙 내려서 데이터 1000개 정도를 화면에 띄워주세요.")
    print("🔥"*20 + "\n")
    
    input("👉 [준비 완료] 스크롤을 다 내렸으면 이 칸에서 엔터(Enter)를 치세요! : ")
    
    print("\n데이터를 긁어오는 중입니다...")
    soup = BeautifulSoup(driver.page_source, 'html.parser')
    rows = soup.find_all('div', class_='body_list')
    
    transactions = []
    for row in rows:
        cols = row.find_all('div', class_='list_txt')
        if len(cols) >= 3:
            size = cols[0].get_text(strip=True)
            price_str = cols[1].get_text(strip=True)
            date_str = cols[2].get_text(strip=True)
            
            if "원" in price_str:
                transactions.append({
                    "date_str": date_str,
                    "size": size,
                    "price_str": price_str
                })
                
    df = pd.DataFrame(transactions)
    driver.quit()
    
    csv_filename = f"kream_transactions_{product_id}.csv"
    
    if not df.empty:
        # 가격 및 날짜 전처리
        df['price_str'] = df['price_str'].astype(str).str.replace(r'[^0-9]', '', regex=True)
        df = df[df['price_str'] != ''] 
        df['price'] = df['price_str'].astype(int)
        
        today = datetime.now()
        def parse_kream_date(d_str):
            d_str = str(d_str)
            if "분 전" in d_str or "시간 전" in d_str: return today.date()
            elif "일 전" in d_str:
                days_ago = int(re.sub(r'[^0-9]', '', d_str))
                return (today - timedelta(days=days_ago)).date()
            else:
                match = re.search(r'\d{2}/\d{2}/\d{2}', d_str)
                if match: return pd.to_datetime("20" + match.group(), format="%Y/%m/%d").date()
                else: return None
                
        df['date'] = df['date_str'].apply(parse_kream_date)
        df = df.dropna(subset=['date']) 
        df['date'] = pd.to_datetime(df['date'])
        df = df.sort_values('date').reset_index(drop=True)
        df = df.drop(columns=['price_str', 'date_str'])

        df.to_csv(csv_filename, index=False, encoding='utf-8-sig')
        print(f"✅ 총 {len(df)}개의 거래 내역을 '{csv_filename}' 파일로 저장 완료했습니다!\n")
        return csv_filename
    else:
        print("❌ 데이터를 찾을 수 없습니다.")
        return None

# ==========================================
# 2. CSV를 읽어 '인기 사이즈' 분석 함수
# ==========================================
def find_popular_size(csv_filename):
    print(f"📊 '{csv_filename}' 파일을 분석합니다...")
    df = pd.read_csv(csv_filename, encoding='utf-8-sig')
    size_counts = df['size'].value_counts()
    top_size = size_counts.idxmax()
    top_count = size_counts.max()
    
    print("\n🏆 [가장 많이 거래된 사이즈 결과] 🏆")
    print(f"👉 1등 황금 사이즈: {top_size} (총 {top_count}건 거래됨)")
    print("-" * 30)
    print("📈 [상위 5개 사이즈 분포]")
    print(size_counts.head(5))
    print("=" * 30 + "\n")

In [13]:
# ==========================================
# 🚀 실행 영역
# ==========================================
product_id = "179981"  # 수집할 크림 상품 번호

# 1. 크롤링 진행
saved_csv = crawl_and_save_csv(product_id)

# 2. 완료 후 인기 사이즈 바로 출력
if saved_csv:
    find_popular_size(saved_csv)

🚀 [179981] 크롬 브라우저를 준비합니다...
🔓 맥 OS 보안 차단 완벽 해제! (경로: /Users/kwagminseo/.wdm/drivers/chromedriver/mac64/144.0.7559.133/chromedriver-mac-arm64/chromedriver)

🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥
🚨 [수동 작업 안내]
1. 팝업창을 닫아주세요.
2. '체결 내역' 창을 클릭해주세요.
3. 마우스 휠을 밑으로 쫙쫙 내려서 데이터 1000개 정도를 화면에 띄워주세요.
🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥



👉 [준비 완료] 스크롤을 다 내렸으면 이 칸에서 엔터(Enter)를 치세요! :  



데이터를 긁어오는 중입니다...
✅ 총 2000개의 거래 내역을 'kream_transactions_179981.csv' 파일로 저장 완료했습니다!

📊 'kream_transactions_179981.csv' 파일을 분석합니다...

🏆 [가장 많이 거래된 사이즈 결과] 🏆
👉 1등 황금 사이즈: W270 (총 316건 거래됨)
------------------------------
📈 [상위 5개 사이즈 분포]
size
W270    316
W275    293
W265    259
W285    177
W240    157
Name: count, dtype: int64

